# 🏃 马拉松训练计划 Agent

从第一性原理出发，结合**工具使用**与**反思**两种 Agentic 设计模式，
构建一个能读取真实跑步数据、生成训练计划、并自我审查调整的 AI Agent。

**闭环流程：**
```
真实跑步数据（文件/手动）→ AI 生成计划 → AI 反思审查 → 输出调整后计划
      (工具使用)              (推理)          (反思)         (最终输出)
```

**技术栈：** GPX/TCX/FIT 文件解析 + 手动输入 JSON + DeepSeek（deepseek-chat 生成 / deepseek-reasoner 反思）

**无需注册任何 API 账号**——从手表 App 导出运动文件或手动填写数据即可。

---
## 阶段 0：第一性原理 —— 理解 Agent 与两大模式

### 什么是 Agent？

Agent 的本质是一个**围绕目标、持续迭代的闭环系统**：

```
感知 → 推理 → 行动 → 反馈 →（再感知...）
```

普通程序是「输入→输出」的一条直线；Agent 多了那条让它**自我修正的反馈回路**。

### 吴恩达四大 Agentic 设计模式

吴恩达提出：**精心设计的 Agentic 工作流，能让弱模型超越强模型**。
他将所有 Agent 工作流归纳为四种模式，本项目使用其中两种：

| 模式 | 本项目落地 | 说明 |
|------|-----------|------|
| **工具使用 Tool Use** | 解析运动文件/读取手动配置 | 让 AI「看见」真实跑步数据，而非凭空生成 |
| **反思 Reflection** | deepseek-reasoner 审查并调整计划 | 让 AI 审视自己的输出并改进 |

### 第一性原理：为什么文件解析也是「工具使用」？

「工具使用」模式的本质是让 AI 通过外部数据感知真实世界。
Garmin API 只是手段之一——用 Python 解析本地运动文件，
同样是让 AI 获取真实跑步数据，本质完全相同。
文件导入无需注册任何 API 账号，所有手表平台都支持导出 GPX/TCX/FIT。

### 本项目的双模型分工

DeepSeek 提供两种模型，天然对应两种角色：

- `deepseek-chat`：标准对话模型 → 扮演**见习教练**，生成初版计划
- `deepseek-reasoner`：带思维链的推理模型 → 扮演**资深审查员**，反思并调整

reasoner 的「思维链」会先逐步推理「这个强度对新手是否过高」，再给出结论——
这种「先想清楚再下结论」的特性，正是反思模式的理想载体。

### 最小闭环设计

```
感知  ──→  推理  ──→  行动  ──→  反馈
取跑步数据    生成计划    输出计划    审查并调整
(data_tool)  (chat)     (输出)    (reasoner)
```

接下来我们一步步把这个闭环跑通。

---
## 阶段 1：环境准备

在开始之前，请确保：
1. 已安装 Python 3.8+
2. 已将 `.env.example` 复制为 `.env`，并填入你的 DeepSeek API Key
3. 准备好数据（二选一）：
   - **方式A**：将运动文件（GPX/TCX/FIT）放入 `data/` 目录
   - **方式B**：复制 `data/manual_input.example.json` 为 `data/manual_input.json`，手动填写跑步数据

### 如何从 Garmin Connect App 导出运动文件？

1. 打开 Garmin Connect App 或网页版（connect.garmin.com）
2. 进入「活动」→ 选择某次跑步记录
3. 点击右上角菜单 → 「导出为 GPX」或「导出为 TCX」
4. 将导出的文件放入项目的 `data/` 目录

GPX/TCX/FIT 三种格式都支持，放多个文件会自动按日期排序取最近 5 次。

In [ ]:
# 安装依赖（首次运行需要，已安装可跳过）
!pip install -r requirements.txt

In [ ]:
# 加载环境变量并验证配置
import os
from dotenv import load_dotenv

load_dotenv()  # 读取 .env 文件

# 验证 DeepSeek API Key 是否配置
assert os.getenv("DEEPSEEK_API_KEY"), "请在 .env 中填写 DEEPSEEK_API_KEY"

print("✅ 环境变量加载成功，配置检查通过")
print(f"   目标完赛时间: {os.getenv('TARGET_FINISH_TIME', '06:00:00')}")

# 检查数据目录状态
import os
data_dir = os.path.join(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'data' else os.getcwd(), 'data')
if os.path.isdir(data_dir):
    files = [f for f in os.listdir(data_dir) if f.endswith(('.gpx', '.tcx', '.fit', '.GPX', '.TCX', '.FIT'))]
    has_manual = os.path.isfile(os.path.join(data_dir, 'manual_input.json'))
    print(f"   数据目录: {data_dir}")
    print(f"   运动文件: {len(files)} 个" + (f" ({', '.join(files)})" if files else ""))
    print(f"   手动配置: {'已存在' if has_manual else '未找到'}")
else:
    print(f"   数据目录尚未创建: {data_dir}（运行后续代码会自动创建）")

---
## 阶段 2：工具使用模式 —— 跑步数据接入

**什么是工具使用模式？**

AI 本身无法直接访问你的运动数据。工具使用模式就是：
由 Python 代码主动解析本地运动文件 / 读取手动配置，把真实数据取回来，再喂给 AI。

**双模式自动选择：**
- **文件模式（优先）**：解析 `data/` 目录下 GPX/TCX/FIT 运动文件
- **手动模式（兜底）**：读取 `data/manual_input.json` 配置
- 有文件用文件，无文件自动回退到手动输入

**关于健康/体能指标：**

运动文件（GPX/TCX/FIT）记录的是单次跑步数据（距离、配速、心率、步频），
不包含日常健康数据（静息心率、睡眠、压力）和高级体能指标（VO2max、训练准备度）。
这些字段在文件模式下会置为 None，可在 `manual_input.json` 中补充。

In [ ]:
# 导入数据工具模块
from data_tool import build_runner_profile, get_recent_runs, get_data_source_info, format_pace
import json

# 查看当前数据源信息
info = get_data_source_info()
print("数据源信息：")
print(json.dumps(info, ensure_ascii=False, indent=2))

In [ ]:
# 工具使用：获取最近 5 次跑步记录
recent_runs = get_recent_runs(limit=5)

print(f"找到 {len(recent_runs)} 次跑步记录：\n")
for run in recent_runs:
    pace = format_pace(run.get("平均配速_秒每公里"))
    print(f"📅 {run['日期']}  |  🏃 {run['距离_km']}km  |  ⏱ {pace}  |  ❤️ 心率{run.get('平均心率')}  |  {run.get('活动名称', '跑步')}")

**构建跑者画像** —— 这是工具使用模式的最终产出。

把分散的 Garmin 数据（跑步记录、健康指标、体能指标）整合成一份
精简、结构化的档案，直接作为 LLM 的输入上下文。

In [ ]:
# 构建完整的跑者画像（汇总跑步数据 + 健康/体能指标）
runner_profile = build_runner_profile()

print("\n跑者画像：")
print(json.dumps(runner_profile, ensure_ascii=False, indent=2))

✅ **验证点**：如果上方能看到你的跑步记录（日期、距离、配速、心率），说明数据接入已跑通。

- 健康指标和体能指标为 None 属正常——运动文件不含日常健康数据
- 如需补充，可编辑 `data/manual_input.json` 中的健康/体能字段

接下来让 AI 基于这些真实数据生成训练计划。

---
## 阶段 3：推理引擎 —— DeepSeek 连通性测试

在生成计划前，先确认 DeepSeek API 可用。

DeepSeek 兼容 OpenAI SDK，调用方式非常简洁。
我们封装了两个核心函数：

| 函数 | 模型 | 作用 |
|------|------|------|
| `generate_training_plan()` | deepseek-chat | 生成训练计划 |
| `reflect_and_adjust()` | deepseek-reasoner | 反思审查并调整 |

In [ ]:
# 导入 LLM 模块并测试连通性
from llm_agent import generate_training_plan, reflect_and_adjust, call_deepseek

# 快速测试：确认 DeepSeek API 可用
test = call_deepseek(
    messages=[{"role": "user", "content": "你好，请回复'连接成功'四个字"}],
    model="deepseek-chat"
)
print(f"DeepSeek 回复：{test['content']}")

✅ **验证点**：如果返回「连接成功」，说明 DeepSeek API 已就绪。

---
## 阶段 4：生成训练计划（推理 + 行动）

这是闭环中的**推理→行动**环节。

AI（deepseek-chat）拿到工具使用模式取回的真实跑者画像，
思考后输出一份 7 天训练计划。

In [ ]:
# 推理：根据跑者画像生成 7 天训练计划
plan = generate_training_plan(runner_profile)

print("=" * 50)
print("📋 AI 生成的训练计划（初版）")
print("=" * 50)
print(plan)

✅ **验证点**：计划应包含 7 天，有休息日，每天有训练类型/距离/配速/心率区间。

但这份计划真的合理吗？对新手来说强度会不会太高？
接下来让 reasoner 来审查。

---
## 阶段 5：反思模式 —— 审查并调整计划（反馈）

这是**反思模式**的体现：让另一个 AI（deepseek-reasoner）审视生成结果。

**为什么用 reasoner？**
- 它自带「思维链」（reasoning_content），会先逐步推理再下结论
- 天然适合「审查」这个需要细致分析的任务

审查重点：
1. 强度是否过高？（新手单次距离、周跑量增幅、心率要求）
2. 恢复是否充足？（休息日数量、连续训练日长度）
3. 配速是否合理？（是否匹配跑者当前能力）

我们会同时展示 reasoner 的**思考过程**（思维链），让你看到 AI 是怎么审查的。

In [ ]:
# 反思：让 reasoner 审查计划并调整
adjusted_plan, reasoning = reflect_and_adjust(plan, runner_profile)

if reasoning:
    print("=" * 50)
    print("🧠 AI 的审查思考过程（思维链）")
    print("=" * 50)
    # 思维链可能很长，只展示前 800 字
    display_text = reasoning[:800] + "...\n（后续省略）" if len(reasoning) > 800 else reasoning
    print(display_text)

In [ ]:
print("=" * 50)
print("📋 反思调整后的训练计划（最终版）")
print("=" * 50)
print(adjusted_plan)

✅ **验证点（反思是否有效）：**

对比初版和调整后的计划，检查：
- 原计划若出现「新手单次距离 > 5km」或「无休息日」或「周跑量增幅 > 10%」，应被修正
- 调整后计划应包含至少 2 个休息日
- 配速要求应在新手有氧区间（心率不超过最大心率 80%）

如果调整后的计划比初版更合理，说明反思模式正在发挥作用！

---
## 阶段 6：完整闭环 —— 马拉松训练 Agent 跑通

把前面所有步骤串成一个完整闭环：

```
感知（取手表数据）→ 推理（生成计划）→ 行动（输出计划）→ 反馈（反思调整）
```

这就是一个最小的、但完整的 Agent 闭环。
运行下方代码，体验从数据到最终计划的一次完整运行。

In [ ]:
"""
马拉松训练 Agent 完整闭环
感知 → 推理 → 行动 → 反馈
"""
from data_tool import build_runner_profile
from llm_agent import generate_training_plan, reflect_and_adjust

# 1. 感知：从本地运动文件/手动配置获取数据
print("【1/4 感知】正在获取跑步数据...")
runner_profile = build_runner_profile()
print(f"  ✅ 已获取 {len(runner_profile['近期跑步数据'])} 次跑步记录\n")

# 2. 推理：生成训练计划
print("【2/4 推理】AI 正在生成训练计划...")
plan = generate_training_plan(runner_profile)
print("  ✅ 计划生成完毕\n")

# 3. 行动：输出初版计划
print("【3/4 行动】初版训练计划：")
print("-" * 40)
print(plan[:300] + "...\n" if len(plan) > 300 else plan)

# 4. 反馈：反思审查并调整
print("【4/4 反馈】AI 正在反思审查并调整...")
final_plan, reasoning = reflect_and_adjust(plan, runner_profile)

print("\n" + "=" * 50)
print("🏃 最终训练计划（经反思调整）")
print("=" * 50)
print(final_plan)

---
## 阶段 7：验证清单与迭代建议

### ✅ 闭环验证清单

| 环节 | 验证方法 | 通过标准 |
|------|---------|----------|
| 环境准备 | 运行阶段 1 | 打印「配置检查通过」，显示数据目录状态 |
| 数据接入 | 运行阶段 2 | 打印出最近跑步记录（文件或手动模式） |
| DeepSeek 连通 | 运行阶段 3 | 返回「连接成功」 |
| 生成计划 | 运行阶段 4 | 输出 7 天结构化计划 |
| 反思调整 | 运行阶段 5 | 能看到思维链；调整后计划更合理 |
| 完整闭环 | 运行阶段 6 | 4 步顺序执行，输出最终计划 |

### 🚀 后续迭代方向

这个 MVP 跑通后，可以逐步增强：

1. **启用 Garmin API（可选）**：如未来注册成功，可启用 `garmin_tool.py` 获取更丰富的健康数据（静息心率、睡眠、VO2max、训练准备度），或回写训练计划到手表日历

2. **多轮反思**：生成→审查→若不通过则再调整，直到满足新手安全标准（设置最大迭代次数防死循环）

3. **趋势分析**：放入更多历史运动文件，分析配速进步趋势

4. **进阶工具使用**：换用支持 Function Call 的模型，让 LLM 自主决定调用哪个工具函数

### 📚 核心概念回顾

- **Agent = 感知 → 推理 → 行动 → 反馈 的闭环**
- **工具使用模式**：代码解析运动文件/读取配置获取真实数据（本质与调用 API 相同）
- **反思模式**：让 AI 审视并改进自己的输出
- **吴恩达的洞察**：工作流比模型大小更重要——好的工作流让弱模型超越强模型

---
🎉 恭喜！你已经从第一性原理出发，构建并跑通了一个最小可用的 Agent！